# 01 · Preprocessing

Inspect → harmonize → clean → optionally reserve an unseen attack → split → fit on train → transform/write each split in batches → release that split.

One dataset per run. Initial loading, splitting, and fitting still need RAM. Saved Parquet files contain ready-to-use transformed features.

## 0. Setup and Imports

In [ ]:
PLATFORM = 'kaggle'  # 'colab' or 'kaggle'
assert PLATFORM in ('colab', 'kaggle')

In [ ]:
if PLATFORM == 'colab':

    from google.colab import drive
    drive.mount('/content/drive')
    INPATH = '/content/drive/MyDrive/Masters/Datasets/NF-UQ-NIDS(Splitted_Dataset)'
    OUTPATH = '/content/drive/MyDrive/Masters/IDS_repo/My repo'

else : 
    INPATH = '/kaggle/input/datasets/omarahmed001/ids-datasets'
    OUTPATH = '/kaggle/working/project'
    

### Optional GPU startup

Run in a fresh kernel before sklearn/fastai imports. Missing GPU/CuPy/cuML gives a printed CPU mode; no GPU libraries are automatically installed. Compatible cuML uses info logs for actual dispatch/fallback. Other supported preprocessing estimators may also accelerate; numerical/tie differences require validation.


In [ ]:
# Run first in a fresh kernel. Compatible cuML/CUDA must already be installed.
%pip install -q "scikit-learn>=1.6" category_encoders imbalanced-learn plotly
import sys
USE_GPU_NEIGHBORS = True
GPU_ACCEL_ENABLED = False
GPU_STATUS = 'CPU (GPU acceleration not requested)'
if USE_GPU_NEIGHBORS:
    try:
        import cupy as cp
        gpu_available = cp.cuda.runtime.getDeviceCount() > 0
    except Exception:
        gpu_available = False
    if not gpu_available:
        GPU_STATUS = 'CPU (NVIDIA GPU/CuPy unavailable)'
    elif 'sklearn' in sys.modules:
        raise RuntimeError('Restart the kernel and run GPU setup before sklearn/fastai imports.')
    else:
        try:
            import cuml.accel
        except ImportError:
            GPU_STATUS = 'CPU (cuML unavailable)'
        else:
            # Installation errors propagate: do not claim CPU fallback after partial activation.
            cuml.accel.install(log_level='info')
            GPU_ACCEL_ENABLED = True
            GPU_STATUS = 'cuML enabled; dispatch logs show actual GPU/CPU execution'
print(GPU_STATUS, flush=True)


In [ ]:
from pathlib import Path
from fastai.tabular.all import df_shrink
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
from sklearn.base import clone
from sklearn.frozen import FrozenEstimator
from sklearn.preprocessing import MinMaxScaler, QuantileTransformer
from sklearn.decomposition import PCA

def format_duration(seconds):
    tenths = int(round(float(seconds) * 10))
    hours, remaining = divmod(tenths, 36_000)
    minutes, remaining = divmod(remaining, 600)
    return f'{hours:02d}h {minutes:02d}m {remaining / 10:04.1f}s'


In [ ]:
DATA_DIR = Path(INPATH)
FILES = {
    'NF_UNSW_NB15': 'NF-UNSW-NB15-v2.parquet',
    'NF_BOT_IOT': 'NF-BoT-IoT-v2.parquet',
    'NF_CSE_CIC_IDS2018': 'NF-CSE-CIC-IDS2018-v2.parquet',
    'NF_TON_IOT': 'NF-ToN-IoT-v2.parquet',
    'CIC-BoT-IoT': 'CIC-BoT-IoT-V2.parquet',
    'CSE-CIC-IDS2018': 'CSE-CIC-IDS2018.parquet',
    'UNSW-NB15': 'UNSW-NB15.parquet'
}

In [ ]:
DATASET = 'CSE-CIC-IDS2018'  # Change this to run one of the other datasets.
HOLD_OUT_ATTACK = False  # False: include every attack class in the 70/15/15 split.
HELD_OUT_ATTACK = 'Worms'  # Used only when HOLD_OUT_ATTACK is True.
SEED = 42
MAX_PER_CLASS = 100_000  # Cap before splitting; retain every smaller-class row.


## 1. Load and inspect

In [ ]:
df = pd.read_parquet(DATA_DIR / FILES[DATASET])
print(DATASET, df.shape)
display(df.head())
# Preserve the original class names.
df = df.rename(columns={'Label': 'Attack'})

# Create the binary target: 0 = benign, 1 = attack.
df['Label'] = (
    df['Attack'].str.strip().str.lower() != 'benign'
).astype('uint8')

display(df[['Label', 'Attack']].head())
display(df.groupby(['Label', 'Attack']).size())
display(df['Label'].value_counts())

In [ ]:
df.Label.value_counts()

In [ ]:
with pd.option_context('display.max_rows', None):
    display(pd.DataFrame({
        'Dtype': df.dtypes,
        'Unique Values': df.nunique()
    }))

- Multiple constant columns that will be dropped later

In [ ]:
df['Down/Up Ratio'].value_counts()

### Actual sample versus the full original dataset

Keep up to `MAX_PER_CLASS = 100_000` randomly selected rows per `Attack`; retain every row from classes smaller than the cap. The cells compare this actual sample with the full original data before encoding, cleaning, splitting, or fitted transformations. No second reference sample or 50,000-row diagnostic cap is used.

The class table measures the deliberate change in class proportions. Numerical features are compared within each Attack using KS distance: 0 means matching empirical CDFs; 0.02 means a maximum CDF gap of 2 percentage points. Nominal fields use total variation distance between category frequencies (0 means matching frequencies). Missing and infinite rates are reported separately. Both distances range from 0 to 1; no universal pass/fail cutoff is imposed. Read the complete tables in `numeric_distribution_report` and `categorical_distribution_report`.

These are descriptive comparisons. KS p-values are not reported because the sample is a subset of the original, many features are discrete, and a small p-value would not establish a practically important difference. Marginal agreement does not prove matching correlations, rare subgroups, or tails. Keeping every minority row gives matching raw distributions for that class; it does not add new independent observations.

`CHECK_SAMPLE_DISTRIBUTION = False` skips the feature comparisons while retaining the class table and releasing the original. Full-data comparisons may take time; they process one column at a time and print each class. Rerun **Load and inspect** before repeating sampling.

Sampling remains before the split in this workflow: validation/test therefore reflect the capped class mix. For evaluation at the original class proportions, split the original data first and cap training only. Preprocessing is still fitted after splitting.

In [ ]:
import gc

# MAX_PER_CLASS is configured above with DATASET.
CHECK_SAMPLE_DISTRIBUTION = True

# Retain the full original data until the comparison below is finished.
# This assignment does not copy the original DataFrame.
original_df = df
original_counts = original_df['Attack'].value_counts(dropna=False)

sampled_train = pd.concat(
    [
        group.sample(n=min(len(group), MAX_PER_CLASS), random_state=SEED)
        for _, group in original_df.groupby(
            'Attack', observed=True, dropna=False
        )
    ],
    ignore_index=True,
)
df = sampled_train.sample(frac=1, random_state=SEED).reset_index(drop=True)
del sampled_train

sample_counts = df['Attack'].value_counts(dropna=False)
class_comparison = pd.DataFrame({
    'Original rows': original_counts,
    'Sample rows': sample_counts,
}).fillna(0).astype('int64').rename_axis('Attack')
class_comparison['Original (%)'] = (
    100 * class_comparison['Original rows'] / len(original_df)
)
class_comparison['Sample (%)'] = (
    100 * class_comparison['Sample rows'] / len(df)
)
class_comparison['Difference (percentage points)'] = (
    class_comparison['Sample (%)'] - class_comparison['Original (%)']
)
class_comparison['Rows retained (%)'] = (
    100 * class_comparison['Sample rows'] / class_comparison['Original rows']
)
print(f'Original: {len(original_df):,} rows | Actual sample: {len(df):,} rows')
display(class_comparison.round(4))
class_tv_distance = (
    class_comparison['Sample (%)'] - class_comparison['Original (%)']
).abs().sum() / 200
print(f'Class-proportion TV distance: {class_tv_distance:.6f}')

In [ ]:
from scipy.stats import ks_2samp

# Nominal fields are compared by category frequencies, not numeric ordering.
# Add another dataset-specific nominal column here if needed.
sampling_nominal_columns = ['Protocol']

if CHECK_SAMPLE_DISTRIBUTION:
    categorical_features = [
        c for c in original_df.columns
        if c not in ('Label', 'Attack')
        and (c in sampling_nominal_columns
             or not pd.api.types.is_numeric_dtype(original_df[c]))
    ]
    numeric_features = [
        c for c in original_df.select_dtypes(include='number').columns
        if c not in ('Label', 'Attack') and c not in categorical_features
    ]

    numeric_results = []
    categorical_results = []

    for attack in original_counts.index:
        print(f'Comparing full original vs actual sample: {attack}', flush=True)
        original_mask = (original_df['Attack'].isna() if pd.isna(attack)
                         else original_df['Attack'].eq(attack))
        sample_mask = (df['Attack'].isna() if pd.isna(attack)
                       else df['Attack'].eq(attack))

        # Use every row in both datasets; process one feature at a time.
        for col in numeric_features:
            a = original_df.loc[original_mask, col].to_numpy(
                dtype=np.float64, na_value=np.nan
            )
            b = df.loc[sample_mask, col].to_numpy(
                dtype=np.float64, na_value=np.nan
            )
            finite_a = a[np.isfinite(a)]
            finite_b = b[np.isfinite(b)]

            distance = (
                ks_2samp(finite_a, finite_b, method='asymp').statistic
                if len(finite_a) and len(finite_b) else np.nan
            )
            numeric_results.append({
                'Attack': attack,
                'feature': col,
                'Original rows': len(a),
                'Sample rows': len(b),
                'KS_D': distance,
                'Original missing (%)': 100 * np.isnan(a).mean(),
                'Sample missing (%)': 100 * np.isnan(b).mean(),
                'Original infinite (%)': 100 * np.isinf(a).mean(),
                'Sample infinite (%)': 100 * np.isinf(b).mean(),
            })
            del a, b, finite_a, finite_b

        for col in categorical_features:
            original_frequencies = original_df.loc[
                original_mask, col
            ].value_counts(normalize=True, dropna=False)
            sample_frequencies = df.loc[
                sample_mask, col
            ].value_counts(normalize=True, dropna=False)
            frequencies = pd.concat([
                original_frequencies.rename('original'),
                sample_frequencies.rename('sample'),
            ], axis=1).fillna(0)
            tv_distance = (
                frequencies['original'] - frequencies['sample']
            ).abs().sum() / 2
            categorical_results.append({
                'Attack': attack,
                'feature': col,
                'TV_distance': tv_distance,
            })
            del original_frequencies, sample_frequencies, frequencies

        del original_mask, sample_mask

    numeric_distribution_report = pd.DataFrame(numeric_results, columns=[
        'Attack', 'feature', 'Original rows', 'Sample rows', 'KS_D',
        'Original missing (%)', 'Sample missing (%)',
        'Original infinite (%)', 'Sample infinite (%)',
    ]).sort_values('KS_D', ascending=False)
    categorical_distribution_report = pd.DataFrame(
        categorical_results, columns=['Attack', 'feature', 'TV_distance']
    ).sort_values('TV_distance', ascending=False)

    print('Largest numerical distribution differences (KS distance):')
    display(numeric_distribution_report.head(30))
    if not numeric_distribution_report.empty:
        display(numeric_distribution_report.groupby('Attack', dropna=False)[
            'KS_D'
        ].agg(['median', 'max']))
    print('Categorical distribution differences (total variation distance):')
    display(categorical_distribution_report)
else:
    print('Within-class distribution comparisons disabled.')

# Release the full original before encoding, splitting, and preprocessing.
# The sampled df and the small comparison reports remain available.
del original_df
gc.collect()
print('Sampling comparison finished; original data released from RAM.')

In [ ]:
# 1. Summarize the feature distances separately for every class.
# Multiply by 100 to express CDF gaps in percentage points.
numeric_summary = (
    numeric_distribution_report
    .groupby('Attack', observed=True, dropna=False)['KS_D']
    .agg(['median', 'max'])
    .mul(100)
    .rename(columns={
        'median': 'Median CDF gap across features (pp)',
        'max': 'Largest CDF gap (pp)',
    })
)

# Start with ALL classes, then attach their numerical results.
sample_distribution_summary = class_comparison[
    ['Original rows', 'Sample rows']
].join(numeric_summary)

print('SUMMARY — one row per class')
print('pp = percentage points; smaller gaps mean closer distributions.')

with pd.option_context(
    'display.max_rows', None,
    'display.max_columns', None,
):
    display(sample_distribution_summary.round(4))


# 2. Show five features for EACH class.
# This limits features per class, not the number of classes displayed.
for attack in sample_distribution_summary.index:
    mask = (
        numeric_distribution_report['Attack'].isna()
        if pd.isna(attack)
        else numeric_distribution_report['Attack'].eq(attack)
    )

    details = (
        numeric_distribution_report.loc[mask, ['feature', 'KS_D']]
        .sort_values('KS_D', ascending=False)
        .head(5)
        .copy()
    )
    details['Max CDF gap (pp)'] = 100 * details['KS_D']

    print(f'\n{attack} — five largest numerical feature differences:')
    display(details.reset_index(drop=True).round(6))


# 3. Show categorical results for ALL classes.
print('\nCATEGORY FREQUENCIES — zero means matching proportions.')

with pd.option_context('display.max_rows', None):
    display(
        categorical_distribution_report
        .sort_values(['Attack', 'feature'])
        .reset_index(drop=True)
        .round(6)
    )

## 2. Encoding

| Feature | Unique values | Type | Handling | 
|---|---:|---|---:|
| `Protocol` | 3 | Nominal | One Hot Encoding
Raw numeric and Boolean fields observed as 0/1 on training bypass encoding, numerical scaling and PCA. Missing binary values use the training mode.


In [ ]:
col_to_one_hot = [
    'Protocol'
]

IMPUTER_FIT_ROWS = 500_000
PCA_FIT_ROWS = IMPUTER_FIT_ROWS  # None: full train; integer: sample cap.
PCA_COMPONENTS = None  # None disables PCA; e.g. 10 reduces scaled measurements to 10 columns.
NUMERIC_SCALER = QuantileTransformer(output_distribution='normal', random_state=SEED)
PREPROCESSING_VERBOSE = True
PRINT_EVERY_BATCHES = 10  # Export reporting only.


In [ ]:
col_categorical = col_to_one_hot 

## 3. Dataset Shrinking

In [ ]:
import warnings
from fastai.tabular.core import df_shrink_dtypes

results = []

for col in df.select_dtypes(include="number").columns:
    # Inspect one column without copying or modifying the DataFrame.
    skip = [c for c in df.columns if c != col]

    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always", RuntimeWarning)

        proposed = df_shrink_dtypes(
            df,
            skip=skip,
            obj2cat=False,
            int2uint=False,
        )

    if caught:
        results.append({
            "column": col,
            "original_dtype": str(df[col].dtype),
            "proposed_dtype": str(proposed.get(col, df[col].dtype)),
            "minimum": df[col].min(),
            "maximum": df[col].max(),
            "warning": "; ".join(str(w.message) for w in caught),
        })

display(pd.DataFrame(results))

In [ ]:
df.columns = df.columns.str.strip()
df['Attack'] = df['Attack'].astype("string[pyarrow]").str.strip()

In [ ]:
df = df_shrink(df, obj2cat=False, int2uint=False)
with pd.option_context('display.max_rows', None):
    display(df.dtypes)

## 4. Clean

In [ ]:
# Identify constant columns
constant_cols = df.columns[df.nunique(dropna=False) == 1]

print("Constant columns:", constant_cols.tolist())
print("Number of constant columns:", len(constant_cols))

# Drop constant columns
#df.drop(columns=constant_cols, inplace=True)

In [ ]:
# Drop constant and unnecessary columns
#df.drop(constant_cols)

In [ ]:
print(df.duplicated().sum(), "fully duplicate rows to remove")

In [ ]:
df.drop_duplicates(inplace=True)
df.reset_index(inplace=True, drop=True)

In [ ]:
# Keep Label and Attack for splitting/evaluation; never feed either to the model.
df = df.loc[:, ~df.columns.str.lower().str.startswith(('unnamed:', 'index'))]
feature_cols = [c for c in df if c not in ('Label', 'Attack')]

# Protocol identifiers, DNS query types, and FTP return codes are nominal categories.
categorical_cols = [c for c in col_categorical if c in feature_cols]
numeric_cols = [c for c in feature_cols if c not in categorical_cols]
df[numeric_cols] = df[numeric_cols].replace([np.inf, -np.inf], np.nan)
print(f'{len(numeric_cols)} numeric and {len(categorical_cols)} categorical features')

In [ ]:
print('Categorical:', categorical_cols)
print('Numeric:', numeric_cols)

In [ ]:
df.replace([np.inf, -np.inf], np.nan, inplace=True)    
print(df.isna().any(axis=1).sum(), "rows with at least one NaN to remove")
# df.dropna(inplace=True)

## 5. Optionally reserve an unseen attack

In [ ]:
# Remove incomplete targets and exact repeated rows before any split.
if HOLD_OUT_ATTACK:
    zero_day = df[df['Attack'].eq(HELD_OUT_ATTACK)]
    known = df[df['Attack'].ne(HELD_OUT_ATTACK)]
    print(f'Eligible: {len(known):,} | held-out {HELD_OUT_ATTACK}: {len(zero_day):,}')
else:
    known = df
    zero_day = df.iloc[:0]  # Keep the same schema; no zero-day file is saved.
    print(f'Holdout disabled: all {len(known):,} rows enter the normal split.')


## 6. Split eligible traffic (70% / 15% / 15%)

In [ ]:
# Stratify eligible traffic by Attack; exclude the held-out attack only when enabled.
train, remainder = train_test_split(
    known, test_size=0.30, stratify=known['Attack'], random_state=SEED
)
val, test = train_test_split(
    remainder, test_size=0.50, stratify=remainder['Attack'], random_state=SEED
)
# Drop intermediate frames; retain only the four clean, untransformed splits.
del df, known, remainder
print('train / val / test / zero-day:', *(len(x) for x in (train, val, test, zero_day)))

## Outliers

In [ ]:
# Only numerical measurements—not categories or flag indicators.
outlier_cols = [
    c for c in numeric_cols
    if c not in col_to_one_hot
    and not c.startswith((
        'TCP_FLAGS_',
        'CLIENT_TCP_FLAGS_',
        'SERVER_TCP_FLAGS_',
    ))
]

rows = []

for col in outlier_cols:
    s = train[col]
    finite = s[np.isfinite(s)]

    q1, median, q3, p99, p999 = finite.quantile(
        [0.25, 0.50, 0.75, 0.99, 0.999]
    )
    iqr = q3 - q1

    # Zero IQR makes this rule uninformative for sparse/zero-heavy features.
    iqr_count = (
        ((finite < q1 - 1.5 * iqr) |
         (finite > q3 + 1.5 * iqr)).sum()
        if iqr > 0 else np.nan
    )

    rows.append({
        'feature': col,
        'missing': s.isna().sum(),
        'infinite': np.isinf(s).sum(),
        'minimum': finite.min(),
        'median': median,
        'p99': p99,
        'p99.9': p999,
        'maximum': finite.max(),
        'IQR': iqr,
        'IQR_flagged': iqr_count,
        'above_p99.9': (finite > p999).sum(),
        'outside_float32': (
            (finite > np.finfo(np.float32).max) |
            (finite < -np.finfo(np.float32).max)
        ).sum(),
        'outside_float64': (
            (finite > np.finfo(np.float64).max) |
            (finite < -np.finfo(np.float64).max)
        ).sum(),
    })

outlier_report = pd.DataFrame(rows).set_index('feature')
with pd.option_context('display.max_rows', None):
    display(outlier_report)

## 8. Fit preprocessing on training data only

This CIC notebook previously had no PCA. `PCA_COMPONENTS = None` preserves its existing numeric outputs. To enable PCA over **all scaled numeric measurement columns**, choose an integer component count (no larger than the number of numeric features and fitting rows). Protocol one-hot columns stay outside PCA.

Set `PCA_FIT_ROWS = IMPUTER_FIT_ROWS` for sampled PCA, `None` for full-training PCA, or another positive cap. Pass these into `make_preprocessor(..., pca_fit_rows=PCA_FIT_ROWS, n_components=PCA_COMPONENTS)`. PCA changes the numeric output columns and requires retraining downstream models.

The CIC imputer and numeric scaler retain their existing sampled fitting behavior, now controlled by `IMPUTER_FIT_ROWS`. One-hot vocabularies include all observed training categories. Learned branches are frozen and assembled with one preview row, avoiding a combined full-training feature matrix. All fitting uses training rows only; transforms/writes remain batched. Full-train PCA mode can still require substantial RAM. Requires scikit-learn >= 1.6.

IterativeImputer uses `skip_complete=True` to skip iterative regressions for complete fitting columns. Other fitting settings and optional numeric PCA remain unchanged.


In [ ]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

def normalize_cic_categories(frame):
    return frame.astype('string').fillna('missing').apply(lambda col: col.str.strip())


def category_fit_frame(data, columns):
    """Fit independent category vocabularies without expanding millions of rows."""
    values = {c: pd.Series(data[c]).drop_duplicates().to_numpy() for c in columns}
    longest = max(map(len, values.values()))
    # Repeat real values to align column lengths; do not invent missing categories.
    return pd.DataFrame({c: np.resize(v, longest) for c, v in values.items()})


def fit_pca(data, columns, preprocessing, n_components=1, fit_rows=None,
            random_state=42, verbose=True):
    """fit_rows=None uses all training rows; an integer caps PCA fitting rows."""
    if fit_rows is not None and fit_rows < 1:
        raise ValueError('fit_rows must be None or a positive row count.')
    if fit_rows is None or fit_rows >= len(data):
        fit_data = data[columns]
    else:
        fit_data = data.sample(n=fit_rows, random_state=random_state)[columns]
    if verbose:
        print(f'PCA fit: {len(fit_data):,}/{len(data):,} training rows; '
              f'{len(columns)} inputs -> {n_components} component(s).', flush=True)
    model = make_pipeline(
        *[step for _, step in preprocessing.steps],
        PCA(n_components=n_components, svd_solver='full'),
        verbose=verbose,
    )
    model.fit(fit_data)
    model.fit_rows_ = len(fit_data)
    if verbose:
        retained = model.steps[-1][1].explained_variance_ratio_.sum()
        print(f'PCA fitted; retained variance: {retained:.2%}.', flush=True)
    return model


def make_preprocessor(train, numeric_cols, one_hot, raw_binary_cols, scaler=None,
                      imputer_fit_rows=500_000, pca_fit_rows=None,
                      n_components=None, random_state=42, verbose=True):
    """Optional PCA over scaled numeric measurements; categories remain separate."""
    sample = train.sample(
        n=min(imputer_fit_rows, len(train)), random_state=random_state,
    )[numeric_cols]
    imputer = IterativeImputer(max_iter=5, random_state=random_state,
                               skip_complete=True, verbose=int(verbose))
    numeric_scaler = StandardScaler() if scaler is None else clone(scaler)
    numbers = make_pipeline(imputer, numeric_scaler, verbose=verbose)
    if verbose:
        print(f'Fitting CIC imputer/scaler on {len(sample):,} training rows...', flush=True)
    numbers.fit(sample)
    imputer_rows = len(sample)
    del sample

    # Preserve the original CIC numeric fit cap. PCA has its own independent cap.
    if n_components is not None:
        numbers_branch = fit_pca(
            train, numeric_cols, preprocessing=make_pipeline(FrozenEstimator(numbers)),
            n_components=n_components, fit_rows=pca_fit_rows,
            random_state=random_state, verbose=verbose,
        )
    else:
        numbers_branch = numbers
        if verbose:
            print('CIC PCA disabled (n_components=None).', flush=True)
    branches = [('numbers', FrozenEstimator(numbers_branch), numeric_cols)]
    if one_hot:
        category_values = normalize_cic_categories(train[one_hot])
        normalize = FunctionTransformer(normalize_cic_categories, feature_names_out='one-to-one')
        categories = make_pipeline(
            normalize.fit(train[one_hot].iloc[:1]),
            OneHotEncoder(handle_unknown='ignore').fit(category_fit_frame(category_values, one_hot)),
        )
        branches.append(('categories', FrozenEstimator(categories), one_hot))
    if raw_binary_cols:
        binary_imputer = SimpleImputer(strategy='most_frequent').fit(train[raw_binary_cols])
        branches.append(('binary_values', FrozenEstimator(binary_imputer), raw_binary_cols))
    result = ColumnTransformer(branches, verbose=verbose)
    result.fit(train.iloc[:1])
    result.imputer_ = imputer
    result.numeric_scaler_ = numeric_scaler
    result.imputer_fit_rows_ = imputer_rows
    result.pca_ = numbers_branch.steps[-1][1] if n_components is not None else None
    if verbose:
        print(f'CIC preprocessor ready: {len(result.get_feature_names_out()):,} features.', flush=True)
    return result


# Detect existing 0/1 fields on TRAIN only; target columns are excluded.
raw_binary_cols = [
    c for c in feature_cols
    if (pd.api.types.is_numeric_dtype(train[c]) or pd.api.types.is_bool_dtype(train[c]))
    and train[c].notna().any()
    and train[c].dropna().isin([0, 1]).all()
]
numeric_cols = [c for c in feature_cols if c not in categorical_cols and c not in raw_binary_cols]
encoding_one_hot = [c for c in categorical_cols if c not in raw_binary_cols]
print('Existing binary columns kept as 0/1:', raw_binary_cols, flush=True)

preprocessor = make_preprocessor(
    train, numeric_cols, encoding_one_hot, raw_binary_cols,
    scaler=NUMERIC_SCALER, imputer_fit_rows=IMPUTER_FIT_ROWS,
    pca_fit_rows=PCA_FIT_ROWS, n_components=PCA_COMPONENTS,
    random_state=SEED, verbose=PREPROCESSING_VERBOSE,
)


Save each split below in batches, then release its original frame. To repeat saving or run the optional raw-data benchmarks afterward, rerun the loading/splitting cells first.

## 9. Transform, save, and free each split

In [ ]:
import pyarrow as pa

feature_names = preprocessor.get_feature_names_out().tolist()
binary_prefixes = ('categories__', 'binary_values__')

original_target_schema = pa.Schema.from_pandas(
    train[['Label', 'Attack']], preserve_index=False
)
print(original_target_schema)
schema = pa.schema(
    [
        (
            name,
            pa.uint8() if name.startswith(binary_prefixes)
            else pa.float32()
        )
        for name in feature_names
    ]
    + [
        original_target_schema.field('Label'),
        original_target_schema.field('Attack'),
    ]
)

print(f'Compact output schema:\n{schema}')

# Output folders are retained for local checkpoint resume.
# Use a new output run rather than deleting existing checkpoints here.


In [ ]:
from datetime import datetime
from time import perf_counter
import gc
import joblib
import sklearn
import pyarrow as pa
import pyarrow.parquet as pq
from scipy.sparse import issparse

PROJECT = Path(OUTPATH)
fitted_imputer = preprocessor.imputer_
fitted_numeric_scaler = preprocessor.numeric_scaler_
fitted_pca = preprocessor.pca_
attack_tag = f"holdout_{HELD_OUT_ATTACK}" if HOLD_OUT_ATTACK else "all_attacks"
RUN_NAME = f"{datetime.now():%Y%m%d_%H%M%S}_{attack_tag}_{type(fitted_imputer).__name__}"
OUTPUT_DIR = PROJECT / 'outputs' / DATASET / 'preprocessing' / RUN_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
BATCH_SIZE = 10000  # Reduce this if a transformed batch still uses too much RAM.
print('Saving fitted preprocessor to:', OUTPUT_DIR, flush=True)
joblib.dump(preprocessor, OUTPUT_DIR / 'preprocessor.joblib', compress=0)
print('Fitted preprocessor saved.', flush=True)


# Exact baseline from the same cleaned, unencoded TRAIN rows used for export.
import json
raw_category_rows = []
for column in encoding_one_hot + raw_binary_cols:
    unique_values = train[column].drop_duplicates()
    raw_category_rows.append({
        'Raw feature': column,
        'Raw unique entries': len(unique_values),
        'Raw values (first 20)': unique_values.head(20).astype(str).tolist(),
        'Raw missing rows': int(train[column].isna().sum()),
    })
    del unique_values
raw_category_lookup = {r['Raw feature']: r for r in raw_category_rows}
encoded_category_rows = []
for column in preprocessor.get_feature_names_out():
    if column.startswith('binary_values__'):
        inputs, encoding = [column.split('__', 1)[1]], 'Raw binary passthrough'
    elif column.startswith('categories__'):
        inputs = [c for c in encoding_one_hot if column.startswith(f'categories__{c}_')]
        encoding = 'One-hot'
    else:
        continue
    encoded_category_rows.append({
        'Raw feature(s)': ' + '.join(inputs),
        'Raw unique entries': {c: raw_category_lookup[c]['Raw unique entries'] for c in inputs},
        'Raw values (preview)': {c: raw_category_lookup[c]['Raw values (first 20)'] for c in inputs},
        'Encoding': encoding,
        'Encoded column': column,
    })
(OUTPUT_DIR / 'raw_categorical_summary.json').write_text(json.dumps({
    'training_rows': len(train),
    'sampling_max_per_class': MAX_PER_CLASS,
    'seed': SEED,
    'raw_features': raw_category_rows,
    'encoded_features': encoded_category_rows,
}, indent=2), encoding='utf-8')
del raw_category_rows, raw_category_lookup, encoded_category_rows

# Move frames into the queue; remove the other references so they can be freed.
splits = {'train': train, 'val': val, 'test': test}
if HOLD_OUT_ATTACK:
    splits['zero_day'] = zero_day
del train, val, test, zero_day
counts = []
binary_export_audit = []
for name in list(splits):
    rows = splits.pop(name)
    split_started = perf_counter()
    total_batches = (len(rows) + BATCH_SIZE - 1) // BATCH_SIZE
    print(f'Starting {name}: {len(rows):,} rows in {total_batches:,} batches.', flush=True)
    with pq.ParquetWriter(OUTPUT_DIR / f'{name}.parquet', schema, compression='zstd') as writer:
        for start in range(0, len(rows), BATCH_SIZE):
            batch = rows.iloc[start:start + BATCH_SIZE]
            X = preprocessor.transform(batch[feature_cols])
            # # Reduce batch memory before densifying sparse output.
            # # float32 rounding is intentional for compact model inputs.
            # X = X.astype(np.float32, copy=False)
            # Densify this small batch only; never the complete split.
            if issparse(X):
                X = X.toarray()
            prepared = pd.DataFrame(X, columns=feature_names)
            prepared['Label'] = batch['Label'].to_numpy()
            prepared['Attack'] = batch['Attack'].to_numpy()
            # Validate BEFORE uint8 casting; no truncation can hide invalid values.
            for column in [c for c in feature_names if c.startswith(binary_prefixes)]:
                values = prepared[column].to_numpy()
                invalid = int((~np.isin(values, [0, 1])).sum())
                raw_column = column.split('__', 1)[1] if column.startswith('binary_values__') else None
                changed = 0
                if raw_column is not None:
                    present = batch[raw_column].notna().to_numpy()
                    changed = int((values[present] != batch.loc[present, raw_column].to_numpy()).sum())
                binary_export_audit.append({'Split': name, 'Column': column,
                    'Route': 'raw binary passthrough' if raw_column is not None else 'categorical encoding output',
                    'Rows checked': len(batch), 'Invalid binary rows': invalid,
                    'Changed nonmissing raw values': changed})
                if invalid or changed:
                    raise ValueError(f'{name}/{column}: binary preprocessing validation failed.')
            table = pa.Table.from_pandas(prepared, schema=schema, preserve_index=False)
            writer.write_table(table)
            batch_number = start // BATCH_SIZE + 1
            if (batch_number == 1 or batch_number == total_batches
                    or batch_number % max(1, PRINT_EVERY_BATCHES) == 0):
                completed = min(start + BATCH_SIZE, len(rows))
                print(f'{name}: batch {batch_number:,}/{total_batches:,}; '
                      f'{completed:,}/{len(rows):,} rows written; '
                      f'{format_duration(perf_counter() - split_started)} elapsed.', flush=True)
            del batch, X, prepared, table
    counts.append(f'- {name}: {len(rows):,} rows')
    print(f'Saved {name}: {len(rows):,} rows in {format_duration(perf_counter() - split_started)}; releasing raw split.', flush=True)
    del rows
    gc.collect()  # Frees Python references; the allocator may retain some reserved RAM.

(OUTPUT_DIR / 'preprocessing_contract.json').write_text(json.dumps({
    'format': 'nf-binary-preserving-voting-v1', 'raw_binary_columns': raw_binary_cols,
    'feature_names': feature_names, 'binary_prefixes': list(binary_prefixes),
    'note': 'Raw 0/1 bypass encoding/scaling/PCA; missing values use train-fitted mode.',
}, indent=2))
binary_preprocessing_audit = pd.DataFrame(binary_export_audit).groupby(
    ['Split', 'Column', 'Route'], as_index=False
)[['Rows checked', 'Invalid binary rows', 'Changed nonmissing raw values']].sum()
binary_preprocessing_audit.to_csv(OUTPUT_DIR / 'binary_preprocessing_audit.csv', index=False)
print('Binary preprocessing audit — all exported rows:')
display(binary_preprocessing_audit)

holdout_note = f"Reserve `{HELD_OUT_ATTACK}` as zero-day data" if HOLD_OUT_ATTACK else "Use all attack classes; no separate zero-day set"
saved_files = '`train.parquet`, `val.parquet`, `test.parquet`' + (', `zero_day.parquet`' if HOLD_OUT_ATTACK else '')
readme = f"""# {DATASET} preprocessing — {RUN_NAME}

Source: `{DATA_DIR / FILES[DATASET]}`
Holdout enabled: `{HOLD_OUT_ATTACK}`. Held-out attack: `{HELD_OUT_ATTACK if HOLD_OUT_ATTACK else "None (all attacks included)"}`. Seed: `{SEED}`. scikit-learn: `{sklearn.__version__}`.

## Sampling and comparison
Before cleaning and splitting, retain up to {MAX_PER_CLASS:,} random rows per Attack class and all rows from smaller classes. Compare the actual sample to the full original using class proportions, within-class numerical KS distance, and categorical total variation. Diagnostics do not use another reference sample. Validation/test reflect the capped class mix.

## Steps followed
1. Inspect data and attack counts; strip column/attack-name whitespace.
2. Remove dataset identifiers, IP addresses, and generated index columns.
3. Encode the configured CIC categories `{col_to_one_hot}` using training-only one-hot vocabularies; replace numeric infinities with NaN.
4. Remove rows missing targets and exact duplicate rows.
5. {holdout_note}; split eligible traffic 70/15/15, stratified by Attack. With holdout disabled, attack classes are eligible for every split; rare classes are not guaranteed to occur in each split.
6. Fit `{fitted_imputer!r}` and `{fitted_numeric_scaler!r}` on {preprocessor.imputer_fit_rows_:,} sampled training rows (cap {IMPUTER_FIT_ROWS:,}). One-hot vocabularies include all observed training categories. Optional numeric PCA: {PCA_COMPONENTS!r} components; requested PCA row cap: {PCA_FIT_ROWS!r}; actual PCA fit rows: {fitted_pca.n_samples_ if fitted_pca is not None else 'disabled'}. PCA variance retained: {fitted_pca.explained_variance_ratio_.sum() if fitted_pca is not None else 'disabled'}. All fitting uses training data only; branches are frozen and assembled with one preview row.
7. Transform each split in batches of at most {BATCH_SIZE:,} rows and write directly to Parquet with Zstandard compression.
8. Release each batch and completed split before continuing.

## Runtime, checkpoints and GPU
Durations use hours/minutes/seconds. Section 7 runs built-in SMOTENC on the prepared training Attack classes, then Tomek Links on the original majority Attack. The default policy selects and targets 0.5 of the largest training class with 3% tolerance. Nothing is resampled in validation or test.
Under resampling/smotenc/, the built-in sampler checkpoint is checked against the prepared training data, category mappings, and policy settings. Save Kaggle outputs before discarding the session.
The three Testing cells compare standard ADASYN, voting categorical ADASYN, and built-in SMOTENC with identical prepared train/validation inputs, class targets, MLP, and optional Tomek policy. They can save reports and full training Parquet files independently under sampler_comparison/.

## Saved files
`preprocessing_contract.json` identifies the binary-preserving export. `binary_preprocessing_audit.csv` checks the saved indicator values.
`raw_categorical_summary.json` records raw training categorical domains and the encoded-column mappings. Section 7 saves `train_smotenc_tomek.parquet` and `smotenc_generated_preview.parquet` with generated samples and Tomek retention flags.
{saved_files} contain TRANSFORMED numeric feature columns plus `Label` and `Attack`.
Features are already imputed, encoded, and scaled: do not transform them again.
`preprocessor.joblib` is for new RAW data only. Run the make_preprocessor cell definitions (including normalize_cic_categories) before loading the joblib object in a fresh session. Feature order is `preprocessor.get_feature_names_out()`.
Storage types: scaled numerical features or optional numeric PCA outputs use float32; one-hot and raw binary features use uint8. Protocol one-hot outputs remain outside PCA. IterativeImputer skips iterative regressions for complete fitting columns.
Label and Attack retain their original Arrow types. The same schema is used for every batch and split.
Float32 reduces precision compared with float64.
These files differ from older exports of cleaned, untransformed features.

{chr(10).join(counts)}

## Resampling and diagnostics
Enable RUN_CATEGORY_DIAGNOSTICS in Testing to compare raw, encoded, and resampled categorical values and inspect generated samples. The comparison separately audits all three samplers, including fractional indicators produced by standard ADASYN.

## Read prepared batches
```python
import pyarrow.parquet as pq
for record_batch in pq.ParquetFile('train.parquet').iter_batches(batch_size=10000):
    batch = record_batch.to_pandas()
    X = batch.drop(columns=['Label', 'Attack'])
    y = batch['Label']
    # Consume this batch directly; do not apply preprocessor.transform(X).
```

Initial loading/splitting and imputer fitting still require RAM. Full-train PCA can require substantial RAM; set PCA_FIT_ROWS to a smaller cap. To reload preprocessor.joblib, use scikit-learn >= 1.6; no custom transformer class is required. Saving releases the original split frames; rerun loading/splitting before using raw-data benchmarks or repeating the save step.
Row splits and exact deduplication do not guarantee session isolation or isolation of identical predictors with conflicting labels.
"""
(OUTPUT_DIR / 'README.md').write_text(readme, encoding='utf-8')
print('Prepared datasets saved to:', OUTPUT_DIR)

## 7. Class Imbalance

The main pipeline prepares and saves the splits, then applies imbalanced-learn’s built-in SMOTENC to the training Attack classes and Tomek Links to the original majority class.

SMOTENC uses the prepared continuous features, including optional numeric PCA outputs. It treats all one-hot columns for a field such as Protocol as a single category. Each raw 0/1 feature is a separate categorical field. The output restores the original feature order and storage types, with both `Attack` and binary `Label`. Use the existing validation and test files for evaluation.

Enable `HANDEL_IMBALANCE` below to run Section 7. Set `RESUME_OUTPUT_DIR` to reuse an existing binary-preserving prepared run. Run the cells in order to load or restore training, apply SMOTENC, save its checkpoint, clean with Tomek, and export the balanced training data.


In [ ]:
HANDEL_IMBALANCE = False  # Enable to run the main SMOTENC + Tomek training pipeline.
SELECT_BELOW_RATIO = 0.50
TARGET_RATIO = 0.50
BALANCE_TOLERANCE = 0.03  # Skip negligible top-ups around the requested target.
REUSE_SMOTENC_CHECKPOINT = True
if not 0 < SELECT_BELOW_RATIO <= TARGET_RATIO <= 1:
    raise ValueError('Require 0 < selection ratio <= target ratio <= 1.')
# Fresh-session resume: set an EXISTING output run here; do not rerun main export.
RESUME_OUTPUT_DIR = None  # e.g. '/kaggle/working/project/outputs/CSE-CIC-IDS2018/preprocessing/<run>'
if RESUME_OUTPUT_DIR is not None:
    OUTPUT_DIR = Path(RESUME_OUTPUT_DIR)


### Shared loading and export tools

Section 7 and the three-method comparison use the same batched loader, category mappings, and Parquet writer.


In [ ]:
from pathlib import Path
from time import perf_counter
import gc
import json
import math
import joblib
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import plotly.express as px
import psutil
import sklearn
import imblearn
from imblearn.under_sampling import TomekLinks

# Load prepared features into one dense matrix in batches.
def read_prepared_matrix(path, feature_names=None):
    parquet = pq.ParquetFile(path)
    schema = parquet.schema_arrow
    names = feature_names or [c for c in schema.names if c not in ('Label', 'Attack')]
    dtype = np.result_type(np.float32, *[
        np.dtype(schema.field(c).type.to_pandas_dtype()) for c in names
    ])
    rows = parquet.metadata.num_rows
    required = rows * (len(names) * dtype.itemsize + np.dtype(object).itemsize)
    available = psutil.virtual_memory().available
    print(f'Loading {rows:,} rows × {len(names):,} features; dtype={dtype}; '
          f'working arrays ≈ {required / 2**30:.2f} GiB; '
          f'available RAM={available / 2**30:.2f} GiB', flush=True)
    if required > available * 0.5:
        raise MemoryError('Working arrays exceed half the available RAM. Reduce training rows.')
    X = np.empty((rows, len(names)), dtype=dtype)
    y = np.empty(rows, dtype=object)
    target_parts = []
    start = 0
    for record in parquet.iter_batches(batch_size=20_000, columns=names + ['Attack', 'Label'], use_threads=False):
        frame = record.to_pandas()
        stop = start + len(frame)
        for j, column in enumerate(names):
            X[start:stop, j] = frame[column].to_numpy(copy=False)
        y[start:stop] = frame['Attack'].to_numpy(copy=False)
        target_parts.append(frame[['Attack', 'Label']].drop_duplicates())
        start = stop
    pairs = pd.concat(target_parts, ignore_index=True).drop_duplicates()
    if pairs['Attack'].duplicated().any():
        raise ValueError('An Attack class has multiple Label values.')
    metadata = {
        'original_columns': schema.names,
        'target_dtypes': pairs.dtypes,
        'attack_to_label': pairs.set_index('Attack')['Label'],
    }
    return X, y, names, metadata

def generated_row_preview(X, y, input_rows, targets, names, label_map):
    positions = []
    for attack in targets:
        positions.extend((input_rows + np.flatnonzero(y[input_rows:] == attack)[:5]).tolist())
    columns = [c for c in names if c.startswith(('categories__', 'binary_values__'))]
    indices = [names.index(c) for c in columns]
    preview = pd.DataFrame(X[np.ix_(positions, indices)], columns=columns)
    preview.insert(0, 'Generated row', positions)
    preview['Attack'] = y[positions]
    preview['Label'] = preview['Attack'].map(label_map)
    preview['Kept after Tomek'] = pd.Series([None] * len(preview), dtype='boolean')
    return preview


def write_resampled_train(path, matrix, labels, prepared_path, label_map, *, fractional=False, model_precision=False):
    """Write the fitted training rows with the prepared feature order and target types."""
    from tempfile import NamedTemporaryFile

    schema = pq.read_schema(prepared_path).remove_metadata()
    features = [name for name in schema.names if name not in ('Label', 'Attack')]
    # The comparison files retain the exact input precision, including fractional ADASYN indicators.
    for index, field in enumerate(schema):
        if field.name in features and (model_precision and pa.types.is_floating(field.type)
                                       or fractional and field.type == pa.uint8()):
            schema = schema.set(index, field.with_type(pa.from_numpy_dtype(matrix.dtype)))

    with NamedTemporaryFile(dir=path.parent, suffix='.parquet.tmp', delete=False) as file:
        temporary = Path(file.name)
    try:
        with pq.ParquetWriter(temporary, schema, compression='zstd') as writer:
            for start in range(0, len(labels), 20_000):
                stop = start + 20_000
                batch = pd.DataFrame(matrix[start:stop], columns=features)
                batch['Attack'] = labels[start:stop]
                batch['Label'] = batch['Attack'].map(label_map)
                writer.write_table(pa.Table.from_pandas(
                    batch, schema=schema, preserve_index=False,
                ))
        temporary.replace(path)
    finally:
        temporary.unlink(missing_ok=True)


### Shared SMOTENC and categorical audit tools


In [ ]:
from imblearn.over_sampling import SMOTENC


def read_category_mappings(output_dir, features):
    contract_path = output_dir / 'preprocessing_contract.json'
    if not contract_path.exists():
        raise ValueError('Rerun preprocessing fit and export to create the binary-preserving prepared files.')
    contract = json.loads(contract_path.read_text())
    if contract.get('format') != 'nf-binary-preserving-voting-v1':
        raise ValueError('Refit and export with binary-preserving preprocessing first.')
    if features != contract['feature_names']:
        raise ValueError('Prepared feature order differs from the preprocessing contract.')
    return json.loads((output_dir / 'raw_categorical_summary.json').read_text())['encoded_features']


def resampling_policy(counts):
    largest = int(counts.max())
    requested = math.ceil(TARGET_RATIO * largest)
    targets = {
        attack: requested for attack, count in counts.items()
        if count < SELECT_BELOW_RATIO * largest
        and count < (1 - BALANCE_TOLERANCE) * requested
    }
    smallest = min((int(counts[attack]) for attack in targets), default=2)
    if smallest < 2:
        raise ValueError('Each targeted Attack class needs at least two training rows.')
    return targets, min(5, smallest - 1)


def build_category_frame(matrix, labels, features, mappings):
    """Represent each logical category by the code of its complete indicator pattern."""
    grouped = {}
    for mapping in mappings:
        column, encoding = mapping['Encoded column'], mapping['Encoding']
        if encoding == 'Joint identifier PCA':
            continue  # PCA scores are continuous, not category indicators.
        key = (column if encoding in ('TCP flag bits', 'Raw binary passthrough')
               else (mapping['Raw feature(s)'], encoding))
        grouped.setdefault(key, {'label': mapping['Raw feature(s)'], 'encoding': encoding,
                                 'columns': []})['columns'].append(column)
    groups = []
    for number, group in enumerate(grouped.values()):
        idx = [features.index(c) for c in group['columns']]
        block = matrix[:, idx]
        if not np.isin(block, [0, 1]).all():
            raise ValueError(f"Prepared categorical indicators are not binary: {group['columns']}")
        if 'one-hot' in group['encoding'].lower() and not (block.sum(axis=1) == 1).all():
            raise ValueError(f"Invalid training one-hot group: {group['label']}")
        patterns, codes = np.unique(block.astype('uint8'), axis=0, return_inverse=True)
        groups.append({**group, 'idx': idx, 'patterns': patterns,
                               'name': f'category_{number:03d}', 'codes': codes})
    if not groups:
        raise ValueError('SMOTENC needs at least one prepared categorical group.')
    categorical_indices = {j for group in groups for j in group['idx']}
    continuous_idx = [j for j in range(len(features)) if j not in categorical_indices]
    if not continuous_idx:
        raise ValueError('SMOTENC needs at least one continuous feature.')
    continuous_names = [features[j] for j in continuous_idx]
    mixed = pd.DataFrame(matrix[:, continuous_idx], columns=continuous_names)
    classes = np.unique(labels)
    for group in groups:
        codes = group.pop('codes')
        mixed[group['name']] = codes
        group['allowed'] = {attack: np.unique(codes[labels == attack])
                            for attack in classes}
    categorical = [group['name'] for group in groups]
    return mixed, groups, continuous_idx, categorical


def restore_and_audit(frame, labels, case, stage, groups, continuous_idx, features, dtype):
    """Check every row before reconstructing the fixed model representation."""
    restored = np.empty((len(frame), len(features)), dtype=dtype)
    restored[:, continuous_idx] = frame[[features[i] for i in continuous_idx]].to_numpy()
    reports = []
    for group in groups:
        codes = frame[group['name']].to_numpy(dtype=float)
        fractional = ~np.isfinite(codes) | (codes != np.floor(codes))
        outside = fractional | (codes < 0) | (codes >= len(group['patterns']))
        unseen_class = 0
        for attack, allowed in group['allowed'].items():
            unseen_class += int((~np.isin(codes[labels == attack], allowed)).sum())
        if outside.any() or unseen_class:
            raise ValueError(f"{case}/{stage}: invalid categorical domain in {group['label']}")
        block = group['patterns'][codes.astype(int)]
        invalid_binary = int((~np.isin(block, [0, 1])).sum())
        invalid_onehot = int((block.sum(axis=1) != 1).sum()) if 'one-hot' in group['encoding'].lower() else 0
        if invalid_binary or invalid_onehot:
            raise ValueError(f"{case}/{stage}: invalid encoded pattern in {group['label']}")
        restored[:, group['idx']] = block
        reports.append({'Case': case, 'Stage': stage, 'Feature': group['label'],
                        'Encoding': group['encoding'], 'Encoded columns': ' | '.join(group['columns']),
                        'Rows checked': len(frame),
                        'Fractional categories': int(fractional.sum()),
                        'Outside training domain': int(outside.sum()),
                        'Unseen within Attack': unseen_class,
                        'Non-binary coordinates': invalid_binary,
                        'Invalid one-hot rows': invalid_onehot})
    if not np.isfinite(restored).all():
        raise ValueError(f'{case}/{stage}: non-finite model input.')
    return restored, pd.DataFrame(reports)


def resample_smotenc(frame, labels, categorical, targets, k, seed):
    if not targets:
        return frame, labels
    return SMOTENC(
        categorical_features=categorical,
        sampling_strategy=targets,
        k_neighbors=k,
        random_state=seed,
    ).fit_resample(frame, labels)


### Checkpoint tools

The checkpoint records the prepared input fingerprint and is written atomically.


In [ ]:
import hashlib

_file_signatures = {}

def prepared_signature(path):
    """Hash once per changed file; content identity survives copying the prepared output run."""
    path = Path(path)
    stat = path.stat()
    key = (str(path.resolve()), stat.st_size, stat.st_mtime_ns)
    if key not in _file_signatures:
        digest = hashlib.sha256()
        with path.open('rb') as stream:
            for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
                digest.update(block)
        _file_signatures[key] = digest.hexdigest()
    return _file_signatures[key]

def atomic_checkpoint(path, state, settings):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix('.tmp')
    joblib.dump({'settings': settings, 'state': state}, temporary, compress=0)
    temporary.replace(path)
    print('Checkpoint saved:', path, flush=True)


### 1. Prepare settings and restore a matching SMOTENC checkpoint

Point `RESUME_OUTPUT_DIR` at the existing prepared run to resume in a fresh session. The built-in sampler uses its own checkpoint file, `smotenc_builtin_checkpoint.joblib`, for the same training data and sampling policy.


In [ ]:
# Run the settings and shared tools first when resuming a prepared run.
SMOTENC_CACHE_LOADED = False
if HANDEL_IMBALANCE:
    train_path = OUTPUT_DIR / 'train.parquet'
    features = [c for c in pq.read_schema(train_path).names if c not in ('Label', 'Attack')]
    category_mappings = read_category_mappings(OUTPUT_DIR, features)
    current_policy_settings = {
        'format': 1, 'sampler': 'imblearn.over_sampling.SMOTENC',
        'train_sha256': prepared_signature(train_path),
        'category_mappings_sha256': prepared_signature(OUTPUT_DIR / 'raw_categorical_summary.json'),
        'features': features, 'dataset': DATASET, 'seed': SEED, 'max_per_class': MAX_PER_CLASS,
        'selection': SELECT_BELOW_RATIO, 'target': TARGET_RATIO,
        'tolerance': BALANCE_TOLERANCE, 'max_neighbors': 5,
        'sklearn': sklearn.__version__, 'imblearn': imblearn.__version__,
    }
    RESAMPLING_DIR = OUTPUT_DIR / 'resampling' / 'smotenc' / f'select_{SELECT_BELOW_RATIO:g}_target_{TARGET_RATIO:g}'
    RESAMPLING_DIR.mkdir(parents=True, exist_ok=True)
    SMOTENC_CHECKPOINT = RESAMPLING_DIR / 'smotenc_builtin_checkpoint.joblib'
    if REUSE_SMOTENC_CHECKPOINT and SMOTENC_CHECKPOINT.exists():
        checkpoint = joblib.load(SMOTENC_CHECKPOINT, mmap_mode='r')
        if checkpoint['settings'] != current_policy_settings:
            raise ValueError('Checkpoint settings/data differ. Set REUSE_SMOTENC_CHECKPOINT=False to recompute.')
        restored_state = checkpoint['state']
        globals().update(restored_state)
        del restored_state, checkpoint
        SMOTENC_CACHE_LOADED = True
        print(f'Restored SMOTENC checkpoint: X={X_smotenc.shape}, y={y_smotenc.shape}', flush=True)
        print('Original SMOTENC compute time:', format_duration(smotenc_seconds), '| backend:', smotenc_backend)
        generated_preview.to_parquet(OUTPUT_DIR / 'smotenc_generated_preview.parquet', index=False)


In [ ]:
if HANDEL_IMBALANCE and not SMOTENC_CACHE_LOADED:
    X_train, y_train, resampling_features, train_metadata = read_prepared_matrix(
        OUTPUT_DIR / 'train.parquet',
    )
    attack_to_label = train_metadata['attack_to_label']
    before_counts = pd.Series(y_train).value_counts().sort_index()
    majority_attack = before_counts.idxmax()
    display(before_counts.rename_axis('Attack').rename('rows').to_frame())
    del train_metadata


### 2. Select minority classes and run SMOTENC

The default policy selects Attack classes below half the largest training class and targets half that count. It skips top-ups within 3% of the target. The output reports training counts, generated rows, and elapsed time.


### SMOTENC — timing, shape and generated-row preview


In [ ]:
if HANDEL_IMBALANCE and not SMOTENC_CACHE_LOADED:
    input_train_rows = len(y_train)
    print(f'Before SMOTENC: X={X_train.shape}, y={y_train.shape}', flush=True)
    smotenc_seconds = 0.0
    sampling_targets, k_neighbors = resampling_policy(before_counts)
    smotenc_skipped = not sampling_targets
    print('Selected class targets:', sampling_targets, flush=True)

    if sampling_targets:
        mixed, groups, continuous_idx, categorical = build_category_frame(
            X_train, y_train, resampling_features, category_mappings,
        )
        smotenc_started = perf_counter()
        print('SMOTENC: starting...', flush=True)
        sampled, y_smotenc = resample_smotenc(
            mixed, y_train, categorical, sampling_targets, k_neighbors, SEED,
        )
        smotenc_seconds = perf_counter() - smotenc_started
        del mixed
        X_smotenc, smotenc_audit = restore_and_audit(
            sampled, y_smotenc, 'smotenc', 'after_sampler', groups,
            continuous_idx, resampling_features, X_train.dtype,
        )
        if not np.array_equal(y_smotenc[:input_train_rows], y_train) or not np.array_equal(
            X_smotenc[:input_train_rows], X_train,
        ):
            raise ValueError('SMOTENC changed or reordered original training rows.')
        del sampled, groups
    else:
        print('No classes require the selected partial-balancing policy; skipping SMOTENC.', flush=True)
        X_smotenc, y_smotenc = X_train, y_train
        smotenc_audit = None

    smotenc_rows = len(y_smotenc)
    print(f'After SMOTENC: X={X_smotenc.shape}, y={y_smotenc.shape}', flush=True)
    print(f'SMOTENC time: {format_duration(smotenc_seconds)} | '
          f'Rows added: {smotenc_rows - input_train_rows:,}', flush=True)
    print(f'Post-SMOTENC feature matrix: {X_smotenc.nbytes / 2**30:.2f} GiB | '
          f'dtype={X_smotenc.dtype}', flush=True)

    smotenc_counts = pd.Series(y_smotenc).value_counts().sort_index()
    print('After SMOTENC (Attack):')
    display(pd.DataFrame({
        'Before': before_counts,
        'After SMOTENC': smotenc_counts,
    }).fillna(0).astype(int).rename_axis('Attack'))

    smotenc_backend = GPU_STATUS if sampling_targets else 'skipped'
    generated_preview = generated_row_preview(
        X_smotenc, y_smotenc, input_train_rows, sampling_targets,
        resampling_features, attack_to_label,
    )
    generated_preview.to_parquet(OUTPUT_DIR / 'smotenc_generated_preview.parquet', index=False)

    del X_train, y_train
    gc.collect()

### 3. Save the SMOTENC checkpoint

The checkpoint contains the oversampled training matrix before Tomek Links, along with the targets, category audit, and timing.


In [ ]:
if HANDEL_IMBALANCE and not SMOTENC_CACHE_LOADED:
    cache_variables = [
        'X_smotenc', 'y_smotenc', 'resampling_features', 'attack_to_label', 'before_counts', 'smotenc_counts',
        'majority_attack', 'input_train_rows', 'smotenc_rows', 'smotenc_seconds',
        'smotenc_skipped', 'generated_preview', 'sampling_targets', 'smotenc_backend', 'smotenc_audit',
    ]
    cache_state = {name: globals()[name] for name in cache_variables}
    atomic_checkpoint(SMOTENC_CHECKPOINT, cache_state, current_policy_settings)
    del cache_state
if HANDEL_IMBALANCE:
    settings_path = RESAMPLING_DIR / 'policy_settings.json'
    settings_path.write_text(json.dumps(current_policy_settings, indent=2))
    if smotenc_audit is not None:
        smotenc_audit.to_csv(RESAMPLING_DIR / 'categorical_audit.csv', index=False)


### 4. Tomek Links — timing and output shape


In [ ]:
if HANDEL_IMBALANCE:
    print(f'Before Tomek Links: X={X_smotenc.shape}, y={y_smotenc.shape}', flush=True)
    # Preserve the original policy: clean only the original majority class.
    tomek = TomekLinks(
        sampling_strategy=[majority_attack],
        n_jobs=-1,
    )
    tomek_started = perf_counter()
    print('Tomek Links: starting...', flush=True)
    X_train_balanced, y_train_balanced = tomek.fit_resample(X_smotenc, y_smotenc)
    tomek_seconds = perf_counter() - tomek_started
    tomek_removed_rows = len(y_smotenc) - len(y_train_balanced)
    print(f'After Tomek Links: X={X_train_balanced.shape}, '
          f'y={y_train_balanced.shape}', flush=True)
    print(f'Tomek Links time: {format_duration(tomek_seconds)} | '
          f'Rows removed: {tomek_removed_rows:,}', flush=True)
    after_counts = pd.Series(y_train_balanced).value_counts()

    # Record which preview rows survived, using Tomek's actual selected indices.
    generated_preview['Kept after Tomek'] = np.isin(
        generated_preview['Generated row'].to_numpy(), tomek.sample_indices_,
    )
    generated_preview.to_parquet(
        OUTPUT_DIR / 'smotenc_generated_preview.parquet', index=False,
    )
    del generated_preview

    # Free the expanded input before plotting or exporting the balanced output.
    del X_smotenc, y_smotenc
    gc.collect()


### 5. Resampling results and timings


In [ ]:
if HANDEL_IMBALANCE:
    distribution = pd.DataFrame({
        'Before': before_counts,
        'After SMOTENC': smotenc_counts,
        'After Tomek Links': after_counts,
    }).fillna(0).astype(int).rename_axis('Attack')

    distribution['Before (%)'] = (
        100 * distribution['Before'] / distribution['Before'].sum()
    ).round(2)
    distribution['After (%)'] = (
        100 * distribution['After Tomek Links']
        / distribution['After Tomek Links'].sum()
    ).round(2)
    display(distribution)

    resampling_summary = pd.DataFrame([
        {'Stage': 'SMOTENC', 'Status': 'cached' if SMOTENC_CACHE_LOADED else ('skipped' if smotenc_skipped else 'finished'),
         'Input rows': input_train_rows, 'Output rows': smotenc_rows,
         'Features': len(resampling_features), 'Time': format_duration(smotenc_seconds)},
        {'Stage': 'Tomek Links', 'Status': 'finished',
         'Input rows': smotenc_rows, 'Output rows': smotenc_rows - tomek_removed_rows,
         'Features': len(resampling_features), 'Time': format_duration(tomek_seconds)},
    ]).set_index('Stage')
    display(resampling_summary)
    distribution_path = RESAMPLING_DIR / 'class_distribution.csv'
    distribution.to_csv(distribution_path)
    timing_path = RESAMPLING_DIR / 'timings.csv'
    resampling_summary.to_csv(timing_path)


### 6. Plot class distribution

Run this cell again to redraw the chart without rerunning Tomek Links.


In [ ]:
if HANDEL_IMBALANCE:
    # Plot aggregated counts instead of passing millions of rows to Plotly.
    plot_counts = distribution[
        ['Before', 'After SMOTENC', 'After Tomek Links']
    ].reset_index().melt(
        id_vars='Attack', var_name='Stage', value_name='Rows'
    )
    plot_counts['Attack'] = plot_counts['Attack'].astype(str)
    fig = px.histogram(
        plot_counts,
        x='Attack',
        y='Rows',
        color='Stage',
        histfunc='sum',
        barmode='group',
        title='Attack class distribution before and after resampling',
        labels={'Rows': 'Number of samples'},
        category_orders={'Attack': distribution.index.astype(str).tolist()},
    )
    fig.update_layout(yaxis_title='Number of samples', xaxis_tickangle=-45)
    plot_path = RESAMPLING_DIR / 'class_distribution.html'
    fig.write_html(plot_path, include_plotlyjs='cdn')
    fig.show()



### 7. Save the balanced training data


In [ ]:
if HANDEL_IMBALANCE:
    balanced_path = OUTPUT_DIR / 'train_smotenc_tomek.parquet'
    write_resampled_train(
        balanced_path, X_train_balanced, y_train_balanced,
        OUTPUT_DIR / 'train.parquet', attack_to_label,
    )
    print('Saved:', balanced_path)

    del X_train_balanced, y_train_balanced
    gc.collect()


Train with `train_smotenc_tomek.parquet`, the existing `val.parquet`, and `test.parquet`. Use `Attack` as the multiclass target or `Label` for binary classification. Exclude both targets from the model inputs. Feed the saved features directly to the model.

The tables and chart show counts before resampling, after SMOTENC, and after Tomek Links.


## Github

# Testing

Run Setup, the dataset settings, and the Section 7 shared tools with `OUTPUT_DIR` pointing to a prepared run. The three-method sampler comparison uses `train.parquet` and `val.parquet`. The categorical diagnostic also reads Section 7 output files when available.


In [ ]:
# Enable the categorical diagnostic to inspect the saved preprocessing and resampling outputs.
RUN_TESTING = True
RUN_CATEGORY_DIAGNOSTICS = True  # Main raw -> encoded -> resampled categorical audit.



### Categorical unique values and generated samples

Enable `RUN_CATEGORY_DIAGNOSTICS` to compare raw training categories, prepared indicator values, and the final Section 7 training output. The tables use every row and display up to `MAX_UNIQUE_EXAMPLES` values per feature. The preview shows up to five generated rows per targeted Attack and their Tomek retention flags. Optional numeric PCA outputs are continuous.

The diagnostic also accepts older Section 7 ADASYN exports. It displays the latest saved three-method audit when available.


In [ ]:
if RUN_CATEGORY_DIAGNOSTICS:
    import json
    import numpy as np
    import pandas as pd
    import pyarrow as pa
    import pyarrow.compute as pc
    import pyarrow.parquet as pq

    # OUTPUT_DIR must point to the same run for all three diagnostic files.
    raw_summary_path = OUTPUT_DIR / 'raw_categorical_summary.json'
    processed_path = OUTPUT_DIR / 'train.parquet'
    resampled_path = OUTPUT_DIR / 'train_smotenc_tomek.parquet'
    preview_path = OUTPUT_DIR / 'smotenc_generated_preview.parquet'
    sampler_name = 'SMOTENC'
    if not resampled_path.exists():
        resampled_path = OUTPUT_DIR / 'train_adasyn_tomek.parquet'
        preview_path = OUTPUT_DIR / 'adasyn_generated_preview.parquet'
        sampler_name = 'ADASYN'
    MAX_UNIQUE_EXAMPLES = 12

    if not all(p.exists() for p in [raw_summary_path, processed_path, resampled_path, preview_path]):
        print('Run Section 7 for a saved training dataset and generated-row preview.')
    else:
        category_baseline = json.loads(raw_summary_path.read_text())
        raw_category_uniques = pd.DataFrame(category_baseline['raw_features'])
        print('Raw categorical values from the SAME training split before processing:')
        display(raw_category_uniques)
        print('Rows compared: raw/processed training = '
              f"{category_baseline['training_rows']:,}; final resampled training = "
              f'{pq.ParquetFile(resampled_path).metadata.num_rows:,}')

        # Read one encoded column at a time, not either complete feature matrix.
        comparison_rows = []
        for mapping in category_baseline['encoded_features']:
            column = mapping['Encoded column']
            before_unique = pc.unique(pq.read_table(
                processed_path, columns=[column]
            ).column(0).cast(pa.float64()))
            after_column = pq.read_table(resampled_path, columns=[column]).column(0).cast(pa.float64())
            after_unique = pc.unique(after_column)
            new_unique = pc.filter(
                after_unique,
                pc.invert(pc.is_in(after_unique, value_set=before_unique)),
            )
            non_binary_rows = None
            non_binary = None
            if mapping['Encoding'] != 'Joint identifier PCA':
                non_binary = int(pc.sum(pc.and_(
                    pc.not_equal(after_unique, 0),
                    pc.not_equal(after_unique, 1),
                )).as_py() or 0)
                non_binary_rows = int(pc.sum(pc.invert(pc.is_in(after_column, value_set=pa.array([0., 1.])))).as_py() or 0)
            comparison_rows.append({
                **mapping,
                'Unique after processing': len(before_unique),
                'Processed values (preview)': before_unique.slice(0, MAX_UNIQUE_EXAMPLES).to_pylist(),
                f'Unique after {sampler_name} + Tomek': len(after_unique),
                'Resampled values (preview)': after_unique.slice(0, MAX_UNIQUE_EXAMPLES).to_pylist(),
                'New unique entries': len(new_unique),
                'New values (preview)': new_unique.slice(0, MAX_UNIQUE_EXAMPLES).to_pylist(),
                'Non-binary unique entries (final)': non_binary,
                'Non-binary ROWS (final)': non_binary_rows,
            })
            del before_unique, after_unique, new_unique, after_column

        category_unique_comparison = pd.DataFrame(comparison_rows)
        print('Before processing -> encoded training -> final resampled training:')
        display(category_unique_comparison)
        print('Indicator columns containing values other than 0 and 1 after resampling:')
        display(category_unique_comparison.loc[
            category_unique_comparison['Non-binary unique entries (final)'].fillna(0).gt(0)
        ].head(30))

        generated_samples = pd.read_parquet(preview_path).rename(columns={'ADASYN row': 'Generated row'})
        if generated_samples.empty:
            print(f'{sampler_name} generated no rows in this run; the generated preview is empty.')
        else:
            encoded_columns = [m['Encoded column'] for m in category_baseline['encoded_features']]
            indicator_columns = [m['Encoded column'] for m in category_baseline['encoded_features']
                                 if m['Encoding'] != 'Joint identifier PCA']
            generated_encoded_values = generated_samples.melt(
                id_vars=['Generated row', 'Attack', 'Label', 'Kept after Tomek'],
                value_vars=indicator_columns,
                var_name='Encoded column', value_name='Generated value',
            )
            generated_fractional_examples = generated_encoded_values.loc[
                ~generated_encoded_values['Generated value'].isin([0, 1])
            ]
            focus_columns = list(dict.fromkeys(
                generated_fractional_examples['Encoded column'].tolist()
                + encoded_columns
            ))[:12]
            print(f'Genuine {sampler_name}-generated rows (up to 5 per oversampled Attack):')
            display(generated_samples[
                ['Generated row', 'Attack', 'Label', 'Kept after Tomek'] + focus_columns
            ].head(20))
            print('Generated non-binary indicator coordinates (first 40):')
            display(generated_fractional_examples.head(40))
            if generated_fractional_examples.empty:
                print('No fractional indicator values occurred in this small preview. '
                      'Use the full-data unique comparison above for all final rows.')

    comparison_root = OUTPUT_DIR / 'sampler_comparison'
    comparison_audits = sorted(comparison_root.glob('*/categorical_audit_all_cases.csv'))
    comparison_audit_path = (
        comparison_audits[-1] if comparison_audits
        else comparison_root / 'categorical_audit_all_cases.csv'
    )
    if comparison_audit_path.exists():
        print('Saved all-row comparison audits (standard ADASYN, voting ADASYN and SMOTENC):')
        display(pd.read_csv(comparison_audit_path))


### Standard ADASYN baseline vs voting categorical ADASYN vs SMOTENC

Run a fresh binary-preserving preprocessing fit and export before these three cells. All methods start from the same prepared training inputs and use the same validation split, multiclass Attack target, class targets and 32-unit ReLU MLP with Adam, a fixed seed and 30 epochs. Equal epochs can give different numbers of updates when the samplers return different row counts.

1. **Original ADASYN baseline** runs imbalanced-learn's standard ADASYN directly on the prepared numeric matrix. Its categorical indicators may become fractional. The audit reports those values and the baseline trains on the original sampler output.
2. **Voting categorical ADASYN** allocates rows by neighborhood difficulty, interpolates continuous features, and takes a vote from the seed's k same-Attack neighbors for each categorical field.
3. **SMOTENC** calls the built-in imbalanced-learn sampler with the same logical groups and targets. Sample generation can use substantial RAM when categorical fields have many distinct values.

The voting methods treat each one-hot group as a single category. Raw 0/1 features are separate categorical fields. Optional numeric PCA components are continuous. Per-field votes can change cross-field combinations.

The optional Tomek switch applies the same original-majority-only policy to all three cases. Audits cover every row immediately after oversampling and after Tomek when enabled. Both voting methods require valid category domains and reconstructed indicators. ADASYN class totals can differ from their requested targets because it rounds the allocation per seed.

With `COMPARISON_SAVE_RESULTS = True`, save validation scores, per-Attack reports, counts, confusion matrices, generated previews, timing, categorical audits, and trained models. Set `COMPARISON_SAVE_DATASETS = True` to also save each method's full model-ready `train.parquet`, after Tomek if enabled. These switches work independently. Each run writes to a new folder under `sampler_comparison/`. The original prepared `val.parquet` and `test.parquet` are shared across methods. Standard ADASYN's export keeps its fractional indicators.


In [ ]:
# 1 / 3 — Original ADASYN baseline and shared comparison setup.
RUN_SAMPLER_COMPARISON = True
COMPARISON_USE_TOMEK = True  # Same original-majority policy for all three cases.
COMPARISON_EPOCHS = 30
COMPARISON_SAVE_RESULTS = True    # Save reports, audits, previews, plots, and models.
COMPARISON_SAVE_DATASETS = False  # Save each method's full training data used by its MLP.

if RUN_SAMPLER_COMPARISON:
    import gc
    import json
    from datetime import datetime
    import joblib
    import numpy as np
    import pandas as pd
    import plotly.express as px
    import psutil
    from time import perf_counter
    from sklearn.neural_network import MLPClassifier
    from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                                 classification_report, confusion_matrix)
    from imblearn.under_sampling import TomekLinks
    from imblearn.over_sampling import ADASYN

    X_compare, y_compare, compare_features, compare_metadata = read_prepared_matrix(
        OUTPUT_DIR / 'train.parquet',
    )
    X_compare_val, y_compare_val, _, _ = read_prepared_matrix(
        OUTPUT_DIR / 'val.parquet', compare_features,
    )
    mappings = read_category_mappings(OUTPUT_DIR, compare_features)
    # Initial estimate only: nearest-neighbor search also needs temporary memory.
    if 3 * X_compare.nbytes + 2 * X_compare_val.nbytes > psutil.virtual_memory().available * 0.8:
        raise MemoryError('Comparison matrices need more headroom; reduce training rows and re-export.')
    compare_counts = pd.Series(y_compare).value_counts().sort_index()
    compare_classes = np.unique(y_compare)
    compare_majority = compare_counts.idxmax()
    compare_label_map = compare_metadata['attack_to_label']
    compare_targets, compare_k = resampling_policy(compare_counts)
    compare_dir = OUTPUT_DIR / 'sampler_comparison' / datetime.now().strftime('%Y%m%d_%H%M%S_%f')
    if COMPARISON_SAVE_RESULTS or COMPARISON_SAVE_DATASETS:
        compare_dir.mkdir(parents=True)
    compare_results, compare_audits = [], []

    mixed_compare, compare_groups, compare_continuous_idx, compare_cat_names = build_category_frame(
        X_compare, y_compare, compare_features, mappings,
    )
    print('Same class targets for all three methods:', compare_targets)
    print(f'Logical categorical fields: {len(compare_cat_names)}; '
          f'expanded categorical vocabulary: {sum(len(g["patterns"]) for g in compare_groups):,}.')

    def audit_standard_adasyn(matrix, labels, stage):
        """Report standard ADASYN's encoded values without repairing the baseline."""
        reports = []
        for group in compare_groups:
            block = matrix[:, group['idx']]
            non_binary = (block != 0) & (block != 1)
            binary_rows = ~non_binary.any(axis=1)
            # Compact keys compare entire 0/1 patterns, including binary-encoded ports.
            width = (len(group['idx']) + 7) // 8
            original = np.packbits(group['patterns'], axis=1).view(f'V{width}').ravel()
            packed = np.packbits(block[binary_rows].astype('uint8'), axis=1).view(f'V{width}').ravel()
            # np.unique sorted the original patterns lexicographically before packing.
            positions = np.minimum(np.searchsorted(original, packed), len(original) - 1)
            known = original[positions] == packed
            actual_codes = positions[known]
            valid_labels = labels[binary_rows][known]
            unseen_class = int((~binary_rows).sum() + (~known).sum())
            for attack, allowed in group['allowed'].items():
                unseen_class += int((~np.isin(actual_codes[valid_labels == attack], allowed)).sum())
            bad_onehot = 0
            if 'one-hot' in group['encoding'].lower():
                bad_onehot = int((~binary_rows | (block.sum(axis=1) != 1)).sum())
            reports.append({'Case': 'standard_adasyn', 'Stage': stage, 'Feature': group['label'],
                            'Encoding': group['encoding'], 'Encoded columns': ' | '.join(group['columns']),
                            'Rows checked': len(matrix), 'Fractional categories': int((block != np.floor(block)).any(axis=1).sum()),
                            'Outside training domain': int((~binary_rows).sum() + (~known).sum()),
                            'Unseen within Attack': unseen_class,
                            'Non-binary coordinates': int(non_binary.sum()), 'Invalid one-hot rows': bad_onehot})
        report = pd.DataFrame(reports)
        compare_audits.append(report)
        if COMPARISON_SAVE_RESULTS:
            report.to_csv(compare_dir / f'standard_adasyn_{stage}_categorical_audit.csv', index=False)

    def evaluate_comparison(case, resample, *, standard=False):
        """A fresh identical MLP per case; validation/test are never resampled."""
        case_dir = compare_dir / case
        if COMPARISON_SAVE_RESULTS or COMPARISON_SAVE_DATASETS:
            case_dir.mkdir()
        started = perf_counter()
        try:
            sampling_started = perf_counter()
            frame, labels = resample()
            sampling_seconds = perf_counter() - sampling_started
            if standard:
                X_fit = frame
                audit_standard_adasyn(X_fit, labels, 'after_sampler')
            else:
                X_fit, audit = restore_and_audit(
                    frame, labels, case, 'after_sampler', compare_groups,
                    compare_continuous_idx, compare_features, X_compare.dtype,
                )
                compare_audits.append(audit)
                if COMPARISON_SAVE_RESULTS:
                    audit.to_csv(compare_dir / f'{case}_after_sampler_categorical_audit.csv', index=False)
            after_sampler = pd.Series(labels).value_counts()
            generated = len(labels) - len(y_compare)
            # Original rows/order must be retained by the oversamplers before cleaning.
            if not np.array_equal(labels[:len(y_compare)], y_compare) or not np.array_equal(X_fit[:len(y_compare)], X_compare):
                raise ValueError('The sampler changed or reordered original rows.')
            if not pd.Series(labels).map(compare_label_map).isin([0, 1]).all():
                raise ValueError('Attack-to-Label mapping is not binary.')
            preview_positions = []
            for attack in compare_targets:
                preview_positions.extend((len(y_compare) + np.flatnonzero(labels[len(y_compare):] == attack)[:5]).tolist())
            preview = pd.DataFrame(X_fit[preview_positions], columns=compare_features)
            preview.insert(0, 'Generated row', preview_positions)
            preview['Attack'] = labels[preview_positions]
            preview['Label'] = preview['Attack'].map(compare_label_map)
            tomek_seconds, removed = 0.0, 0
            if COMPARISON_USE_TOMEK:
                cleaning_started = perf_counter()
                print(case, 'Tomek Links: starting...', flush=True)
                cleaner = TomekLinks(sampling_strategy=[compare_majority], n_jobs=-1)
                X_fit, labels = cleaner.fit_resample(X_fit, labels)
                tomek_seconds = perf_counter() - cleaning_started
                removed = len(frame) - len(labels)
                preview['Kept after Tomek'] = np.isin(preview_positions, cleaner.sample_indices_)
                if standard:
                    audit_standard_adasyn(X_fit, labels, 'after_tomek')
                else:
                    # Check retained logical domains again, using all retained rows.
                    retained = frame.iloc[cleaner.sample_indices_]
                    checked, audit = restore_and_audit(
                        retained, labels, case, 'after_tomek', compare_groups,
                        compare_continuous_idx, compare_features, X_compare.dtype,
                    )
                    compare_audits.append(audit)
                    if COMPARISON_SAVE_RESULTS:
                        audit.to_csv(compare_dir / f'{case}_after_tomek_categorical_audit.csv', index=False)
                    if not np.array_equal(checked, X_fit):
                        raise ValueError('Tomek altered retained feature values.')
                    del checked, retained
                del cleaner
            else:
                preview['Kept after Tomek'] = True
            if COMPARISON_SAVE_RESULTS:
                preview.to_parquet(case_dir / 'generated_preview.parquet', index=False)
            if COMPARISON_SAVE_DATASETS:
                write_resampled_train(
                    case_dir / 'train.parquet', X_fit, labels, OUTPUT_DIR / 'train.parquet',
                    compare_label_map, fractional=standard, model_precision=True,
                )
            counts = pd.DataFrame({'Before': compare_counts, 'After sampler': after_sampler,
                                   'After cleaning': pd.Series(labels).value_counts()}).fillna(0).astype(int)
            if COMPARISON_SAVE_RESULTS:
                counts.rename_axis('Attack').to_csv(case_dir / 'class_counts.csv')
            print(f'{case}: training the same 32-unit MLP for {COMPARISON_EPOCHS} epochs; X={X_fit.shape}', flush=True)
            model = MLPClassifier(hidden_layer_sizes=(32,), activation='relu', solver='adam',
                                  batch_size=min(1024, len(labels)), learning_rate_init=0.001,
                                  alpha=1e-4, random_state=SEED)
            fit_started = perf_counter()
            for epoch in range(COMPARISON_EPOCHS):
                model.partial_fit(X_fit, labels, classes=compare_classes)
            fit_seconds = perf_counter() - fit_started
            prediction = model.predict(X_compare_val)
            report = pd.DataFrame(classification_report(y_compare_val, prediction, labels=compare_classes,
                                                        output_dict=True, zero_division=0)).T
            if COMPARISON_SAVE_RESULTS:
                report.to_csv(case_dir / 'per_attack_report.csv')
            cm = confusion_matrix(y_compare_val, prediction, labels=compare_classes)
            if COMPARISON_SAVE_RESULTS:
                pd.DataFrame(cm, index=compare_classes, columns=compare_classes).to_csv(case_dir / 'confusion_matrix.csv')
            fig = px.imshow(cm, x=compare_classes, y=compare_classes, text_auto=True,
                            labels={'x': 'Predicted Attack', 'y': 'True Attack'}, title=case)
            if COMPARISON_SAVE_RESULTS:
                fig.write_html(case_dir / 'confusion_matrix.html', include_plotlyjs='cdn')
            fig.show()
            if COMPARISON_SAVE_RESULTS:
                joblib.dump(model, case_dir / 'model.joblib')
            row = {'Case': case, 'Status': 'finished', 'Training rows': len(labels),
                   'Generated rows': generated, 'Tomek removed': removed,
                   'Accuracy': accuracy_score(y_compare_val, prediction),
                   'Balanced accuracy': balanced_accuracy_score(y_compare_val, prediction),
                   'Macro F1': f1_score(y_compare_val, prediction, labels=compare_classes, average='macro', zero_division=0),
                   'Weighted F1': f1_score(y_compare_val, prediction, labels=compare_classes, average='weighted', zero_division=0),
                   'Sampler time': format_duration(sampling_seconds), 'Tomek time': format_duration(tomek_seconds),
                   'Training time': format_duration(fit_seconds), 'Total time': format_duration(perf_counter() - started)}
            if COMPARISON_SAVE_RESULTS:
                settings = {
                    'metrics': row, 'targets': compare_targets, 'seed': SEED,
                    'epochs': COMPARISON_EPOCHS,
                    'Tomek target': compare_majority if COMPARISON_USE_TOMEK else None,
                    'prepared validation': '../../../val.parquet',
                    'prepared test': '../../../test.parquet',
                }
                (case_dir / 'metrics.json').write_text(json.dumps(settings, indent=2))
            display(counts)
            display(report)
        except (ValueError, RuntimeError, MemoryError) as error:
            row = {'Case': case, 'Status': 'failed', 'Error': str(error)}
            if COMPARISON_SAVE_RESULTS:
                (case_dir / 'error.json').write_text(json.dumps(row, indent=2))
            print(case, 'failed:', error, flush=True)
        compare_results.append(row)
        if COMPARISON_SAVE_RESULTS:
            pd.DataFrame(compare_results).to_csv(compare_dir / 'comparison.csv', index=False)
        gc.collect()

    def standard_adasyn():
        if not compare_targets:
            return X_compare, y_compare
        return ADASYN(sampling_strategy=compare_targets, n_neighbors=compare_k,
                      random_state=SEED).fit_resample(X_compare, y_compare)

    evaluate_comparison('standard_adasyn', standard_adasyn, standard=True)


In [ ]:
# 2 / 3 — Custom ADASYN allocation + SMOTENC-style neighbor voting.
if RUN_SAMPLER_COMPARISON:
    from scipy import sparse
    from sklearn.neighbors import NearestNeighbors
    from sklearn.preprocessing import OneHotEncoder

    def vote_category(neighbor_codes, rng):
        """Pick a mode, giving each tied category an equal chance."""
        # Count votes for each category, then mark a single slot per tied winner.
        votes = np.sort(neighbor_codes, axis=1)
        frequency = (votes[:, :, None] == votes[:, None, :]).sum(axis=2)
        first = np.ones(votes.shape, dtype=bool)
        first[:, 1:] = votes[:, 1:] != votes[:, :-1]
        winners = first & (frequency == frequency.max(axis=1, keepdims=True))
        # Choose uniformly among the unique winners for each synthetic row.
        choice = (rng.random_sample(len(votes)) * winners.sum(axis=1)).astype(int) + 1
        selected = (winners & (winners.cumsum(axis=1) == choice[:, None])).argmax(axis=1)
        return votes[np.arange(len(votes)), selected]

    def categorical_adasyn_voting(frame, labels, categorical, targets, k=5, seed=42):
        """Experimental categorical ADASYN; modes use ALL k same-class neighbors."""
        if not targets:
            return frame, labels
        rng = np.random.RandomState(seed)
        continuous = [c for c in frame if c not in categorical]
        numerical = frame[continuous].to_numpy()
        # Category codes are identifiers. One-hot encode them for distance so their numerical order has no effect.
        categories = frame[categorical].to_numpy(dtype=np.int64)
        encoder = OneHotEncoder(handle_unknown='ignore', dtype=numerical.dtype)
        encoded_categories = encoder.fit_transform(categories).tocsr()
        frame_parts, label_parts = [frame], [labels]
        for attack, target in sorted(targets.items()):
            original_positions = np.flatnonzero(labels == attack)
            needed = int(target) - len(original_positions)
            if needed <= 0:
                continue
            if len(original_positions) <= k:
                raise ValueError(f'{attack}: need at least k+1 training examples.')
            # Same categorical mismatch weighting used by SMOTENC.
            weight = np.median(np.std(numerical[original_positions], axis=0)) / np.sqrt(2)
            distance_matrix = sparse.hstack([
                sparse.csr_matrix(numerical), encoded_categories * weight,
            ], format='csr')
            all_neighbors = NearestNeighbors(n_neighbors=k + 1, n_jobs=-1).fit(distance_matrix)
            nearest = all_neighbors.kneighbors(distance_matrix[original_positions], return_distance=False)
            # Exclude the actual row itself, including when exact-distance ties occur.
            nearest = np.asarray([row[row != i][:k] for row, i in zip(nearest, original_positions)])
            # Use all Attack classes here. Seeds near other classes receive more synthetic rows.
            difficulty = (labels[nearest] != attack).mean(axis=1)
            if not difficulty.sum():
                raise ValueError(f'{attack}: no other-class neighbors; ADASYN is undefined here.')
            # Convert the difficulty weights to per-seed counts. Rounding can shift the class total.
            allocation = np.rint(needed * (difficulty / difficulty.sum())).astype(int)
            seed_rows = np.repeat(np.arange(len(original_positions)), allocation)
            if not len(seed_rows):
                raise ValueError(f'{attack}: all ADASYN allocations rounded to zero.')
            # Restrict synthesis to the current Attack after allocating rows using the full training set.
            same_neighbors = NearestNeighbors(n_neighbors=k + 1, n_jobs=-1).fit(distance_matrix[original_positions])
            nearby = same_neighbors.kneighbors(distance_matrix[original_positions], return_distance=False)
            nearby = np.asarray([row[row != i][:k] for i, row in enumerate(nearby)])
            # Choose one same-Attack neighbor and one interpolation fraction per synthetic row.
            partner = nearby[seed_rows, rng.randint(k, size=len(seed_rows))]
            fraction = rng.uniform(size=(len(seed_rows), 1))
            A = numerical[original_positions[seed_rows]]
            B = numerical[original_positions[partner]]
            synthetic = pd.DataFrame(A + fraction * (B - A), columns=continuous)
            # Vote over all k same-Attack neighbors for each whole category.
            # The randomly chosen interpolation partner gets the same vote as every neighbor.
            voting_neighbors = original_positions[nearby[seed_rows]]
            for j, column in enumerate(categorical):
                synthetic[column] = vote_category(categories[voting_neighbors, j], rng)
            # Restore the mixed-frame column order and keep original rows first for the audit and preview.
            frame_parts.append(synthetic[frame.columns])
            label_parts.append(np.full(len(synthetic), attack, dtype=labels.dtype))
            print(f'Categorical ADASYN: {attack}; requested {needed:,}, generated {len(synthetic):,}', flush=True)
            del distance_matrix, all_neighbors, same_neighbors, synthetic
        return pd.concat(frame_parts, ignore_index=True), np.concatenate(label_parts)

    evaluate_comparison('adasyn_voting', lambda: categorical_adasyn_voting(
        mixed_compare, y_compare, compare_cat_names, compare_targets,
        k=compare_k, seed=SEED,
    ))


In [ ]:
# 3 / 3 — SMOTENC, categorical checks, and the three-way summary.
if RUN_SAMPLER_COMPARISON:
    evaluate_comparison('smotenc', lambda: resample_smotenc(
        mixed_compare, y_compare, compare_cat_names, compare_targets, compare_k, SEED,
    ))

    sampler_comparison = pd.DataFrame(compare_results)
    print('Identical prepared validation rows, feature order, architecture, seed, and epoch budget:')
    display(sampler_comparison)
    if compare_audits:
        categorical_comparison_audit = pd.concat(compare_audits, ignore_index=True)
        if COMPARISON_SAVE_RESULTS:
            categorical_comparison_audit.to_csv(compare_dir / 'categorical_audit_all_cases.csv', index=False)
        checks = ['Fractional categories', 'Outside training domain', 'Unseen within Attack',
                  'Non-binary coordinates', 'Invalid one-hot rows']
        print('Categorical audit totals summed across groups (a row can count in multiple groups):')
        display(categorical_comparison_audit.groupby(['Case', 'Stage'])[checks].sum())
    successful = sampler_comparison.loc[sampler_comparison['Status'].eq('finished')]
    if not successful.empty:
        score_plot = successful.melt(id_vars='Case', value_vars=['Balanced accuracy', 'Macro F1', 'Weighted F1'],
                                     var_name='Metric', value_name='Score')
        fig = px.bar(score_plot, x='Case', y='Score', color='Metric', barmode='group',
                     title='Standard ADASYN vs categorical ADASYN voting vs SMOTENC — validation')
        if COMPARISON_SAVE_RESULTS:
            fig.write_html(compare_dir / 'comparison.html', include_plotlyjs='cdn')
        fig.show()
    if COMPARISON_SAVE_RESULTS or COMPARISON_SAVE_DATASETS:
        print('Comparison outputs:', compare_dir)
    print('Category-domain validity does not prove realistic cross-feature combinations. '
          'Optional numeric PCA outputs are continuous.')
    del X_compare, X_compare_val, mixed_compare
    gc.collect()
